# Flow Cytometry Shuffled Inference Summary

This notebook summarizes rejection rates from shuffled edge-inference results stored in `flow_cytometry_inference/results_shuffled`.

Rejection rate is computed as the fraction of valid p-values with `p_value <= 0.05`.


In [18]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

RESULTS_DIR = Path('results_shuffled')
ALPHA = 0.05
pattern = re.compile(r'(?P<edge>.+)_n(?P<n>\d+)_rep(?P<rep>\d+)_(?P<scaler>qt|std)\.csv$')

files = sorted(RESULTS_DIR.glob('*.csv'))
print(f'Found {len(files)} CSV files in {RESULTS_DIR}')
for f in files:
    print('-', f.name)


Found 16 CSV files in results_shuffled
- p44_42_to_pakts473_shuffled_n1000_rep100_qt.csv
- p44_42_to_pakts473_shuffled_n100_rep100_qt.csv
- p44_42_to_pakts473_shuffled_n200_rep100_qt.csv
- p44_42_to_pakts473_shuffled_n500_rep100_qt.csv
- pip2_plcg_pkc_shuffled_n1000_rep100_qt.csv
- pip2_plcg_pkc_shuffled_n100_rep100_qt.csv
- pip2_plcg_pkc_shuffled_n200_rep100_qt.csv
- pip2_plcg_pkc_shuffled_n500_rep100_qt.csv
- pip3_pakts473_shuffled_n1000_rep100_qt.csv
- pip3_pakts473_shuffled_n100_rep100_qt.csv
- pip3_pakts473_shuffled_n200_rep100_qt.csv
- pip3_pakts473_shuffled_n500_rep100_qt.csv
- pkc_pka_shuffled_n1000_rep100_qt.csv
- pkc_pka_shuffled_n100_rep100_qt.csv
- pkc_pka_shuffled_n200_rep100_qt.csv
- pkc_pka_shuffled_n500_rep100_qt.csv


In [19]:
rows = []
for f in files:
    m = pattern.match(f.name)
    if m is None:
        continue

    df = pd.read_csv(f)
    if 'p_value' not in df.columns:
        continue

    pvals = pd.to_numeric(df['p_value'], errors='coerce')
    valid = pvals.notna()
    n_valid = int(valid.sum())
    reject_count = int((pvals[valid] <= ALPHA).sum()) if n_valid > 0 else 0
    reject_rate = (reject_count / n_valid) if n_valid > 0 else np.nan

    rows.append({
        'file': f.name,
        'edge': m.group('edge'),
        'n': int(m.group('n')),
        'n_rep_from_filename': int(m.group('rep')),
        'scaler': m.group('scaler'),
        'rows_in_csv': int(len(df)),
        'n_valid_pvalues': n_valid,
        'reject_count_alpha_005': reject_count,
        'rejection_rate_alpha_005': reject_rate,
    })

summary_df = pd.DataFrame(rows).sort_values(['edge', 'n', 'scaler']).reset_index(drop=True)
summary_df


,file,edge,n,n_rep_from_filename,scaler,rows_in_csv,n_valid_pvalues,reject_count_alpha_005,rejection_rate_alpha_005
0,p44_42_to_pakts473_shuffled_n100_rep100_qt.csv,p44_42_to_pakts473_shuffled,100,100,qt,100,100,86,0.86
1,p44_42_to_pakts473_shuffled_n200_rep100_qt.csv,p44_42_to_pakts473_shuffled,200,100,qt,100,100,97,0.97
2,p44_42_to_pakts473_shuffled_n500_rep100_qt.csv,p44_42_to_pakts473_shuffled,500,100,qt,100,100,100,1.00
3,p44_42_to_pakts473_shuffled_n1000_rep100_qt.csv,p44_42_to_pakts473_shuffled,1000,100,qt,100,100,100,1.00
4,pip2_plcg_pkc_shuffled_n100_rep100_qt.csv,pip2_plcg_pkc_shuffled,100,100,qt,100,100,7,0.07
5,pip2_plcg_pkc_shuffled_n200_rep100_qt.csv,pip2_plcg_pkc_shuffled,200,100,qt,100,100,7,0.07
6,pip2_plcg_pkc_shuffled_n500_rep100_qt.csv,pip2_plcg_pkc_shuffled,500,100,qt,100,100,5,0.05
7,pip2_plcg_pkc_shuffled_n1000_rep100_qt.csv,pip2_plcg_pkc_shuffled,1000,100,qt,100,100,5,0.05
8,pip3_pakts473_shuffled_n100_rep100_qt.csv,pip3_pakts473_shuffled,100,100,qt,100,100,4,0.04
9,pip3_pakts473_shuffled_n200_rep100_qt.csv,pip3_pakts473_shuffled,200,100,qt,100,100,5,0.05


In [20]:
pivot = summary_df.pivot_table(
    index=['edge', 'scaler'],
    columns='n',
    values='rejection_rate_alpha_005',
    aggfunc='mean'
).sort_index()
pivot


,n,100,200,500,1000
edge,scaler,,,,
p44_42_to_pakts473_shuffled,qt,0.86,0.97,1.00,1.00
pip2_plcg_pkc_shuffled,qt,0.07,0.07,0.05,0.05
pip3_pakts473_shuffled,qt,0.04,0.05,0.07,0.04
pkc_pka_shuffled,qt,0.05,0.05,0.09,0.07


In [21]:
out_csv = RESULTS_DIR / 'shuffled_rejection_rate_summary.csv'
summary_df.to_csv(out_csv, index=False)
print(f'Saved summary table to: {out_csv}')


Saved summary table to: results_shuffled/shuffled_rejection_rate_summary.csv
